In [1]:
from sympy.physics.units import temperature
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, AutoModelForCausalLM
import torch
import json
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import os
import re


In [2]:
# works good with chat template
model_name = "Qwen/Qwen2-1.5B-Instruct"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, )
model = AutoModelForCausalLM.from_pretrained(model_name,trust_remote_code=True, torch_dtype= torch.float16, device_map='auto' )

# print(model.device)
# print("Memory:", model.get_memory_footprint())

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [4]:
# depend on reponse format / structure output for accurate reponse

def build_tone_classifier_prompt(user_input, context):
    response_format = """{
    "tone":"<String>",
    "confidence":"<int>",
    }"""
    return f"""
You are a tone classification system.

Your task is to classify the tone of the given message.

Allowed labels:
[angry, abusive, love, confused, sarcasm, frustrated, ignorance]

STRICT RULES:
- Output ONLY one word from the labels
- Do NOT explain
- Do NOT repeat input
- Do NOT include context
- Do NOT add any extra text
- Do NOT return Note
- Output must be in response format
- Do NOT repeat the output

Message:
{user_input}

Context (for internal understanding only, NEVER include in output):
{context}

Response Format:
{response_format}
"""

In [5]:
knowledge_base = ""
with open("knowledge/tone_rules.md", "r", encoding="utf-8") as f:
    knowledge_base = f.read()

In [6]:
# RAG Using Faiss

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
sentence_model_prev = SentenceTransformer("all-MiniLM-L6-v2")
document = []
data = []



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
user_input = """ 
Amit: When are we having the call???
"""

In [8]:

import re

def chunk_text(text):
    # split by markdown headings like ## Anger, ## Sadness
    chunks = re.split(r"\n## ", text)
    
    # clean and re-add heading
    cleaned_chunks = []
    for i, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if not chunk:
            continue
        
        if i != 0:
            chunk = "## " + chunk  # add back heading
        
        cleaned_chunks.append(chunk)
    
    return cleaned_chunks


In [9]:
def build_index(document):
    emb = sentence_model.encode(document)
    dim = emb.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(emb))
    return index


In [10]:
def retrive(index,document,query,k=5):
    q_emb = sentence_model.encode([query],)
    D, I = index.search(q_emb,k, )
    results = []
    for dist, ind in zip(D[0],I[0]):
        if dist >= 0.5:
            results.append(document[ind])
    return results[ind]

In [ ]:
def extract_json_from_text(text):
    try:
        # Find JSON pattern using regex
        match = re.search(r'\{.*?\}', text)
        
        if match:
            json_str = match.group()
            return json.loads(json_str)
        
    except json.JSONDecodeError:
        pass
    
    return None


def get_tone(user_input, tone_context):
    prompt = build_tone_classifier_prompt(user_input, tone_context)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.3,
        do_sample=True,
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    tone = extract_json_from_text(resp)
    return tone

In [12]:
def save_json(message):
    with open("data/dataset.jsonl", "a", encoding="utf-8", ) as f:
        json.dump(message, f)
        f.write("\n")

In [13]:
# fetch last 24hrs conversation
data.clear()
with open("data/previ_chats.jsonl", "r") as f:
    for i, line in enumerate(f):
        line = line.strip()
        
        if not line:
            continue  # skip empty lines
        
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error on line {i}: {line}")
            print("Reason:", e)

def build_index_prev_conv():
    texts = [
        f"{m.get('sender','unknown')}: {m.get('text','')}"
        for m in data
    ]
    emb = sentence_model_prev.encode(texts)
    emb = np.array(emb).astype("float32")
    dimm = emb.shape[1]
    # print("Embedding type:", type(emb))
    # print("Embedding shape:", emb.shape)
    msgs_index = faiss.IndexFlatL2(dimm)
    msgs_index.add(np.array(emb))
    return msgs_index




In [14]:
# retrive from previ msg the best fix

def retrive_prev_msg(msgs_index, query,k=5,):
    q_emb = sentence_model_prev.encode([query],)
    D, I = msgs_index.search(q_emb,5, )
    recent_msgs = []
    for dist, ind in zip(D[0],I[0]):
        # if dist >= 0.5:
        recent_msgs.append(data[ind])
    return recent_msgs

In [15]:
def build_user_prompt(user_input, tone):
    msg_index = build_index_prev_conv()
    retrived_data = retrive_prev_msg(msg_index, user_input, 5)
    formatted_context = "\n".join([
        f"{m['sender']}: {m['text']}" for m in retrived_data
    ])
    # context = retrive(user_input)
    print(f"userInput: {user_input}\nPrevious Chats pickted: {formatted_context}\n\nTONE: {tone}")
    return f"""
    You are acting as Karan. Your job is to reply exactly like Karan would.

IMPORTANT:
- Messages from "karan" are your past replies → learn the style from them
- Messages from others are incoming messages
- Use past messages to understand context, commitments, and tone

STYLE GUIDELINES (based on Karan):
- Short and clear
- Professional but friendly
- Uses phrases like: "Sure", "Let me check", "I'll confirm"


CONVERSATION HISTORY:
{formatted_context}

CURRENT MESSAGE (you need to reply to this):
{user_input}

INCOMING MESSAGE TONE:
{tone}

INSTRUCTIONS:
- If there was a previous commitment (e.g., "evening") → use ONLY that wording
- DO NOT generate exact times unless explicitly present in conversation
- If exact time is not confirmed → say you will check and confirm
- If this is a follow-up → acknowledge and respond accordingly
- Keep it concise and natural

CRITICAL RULE:
- Do NOT invent or assume specific times (e.g., "8 PM", "7:30")
- Only use time references explicitly mentioned in conversation (e.g., "evening")
- If exact time is not known → say you will confirm

Generate ONLY the reply as Karan:
    """



In [16]:
def get_knowedgebase_context(user_input):
    chunked_data = chunk_text(knowledge_base)
    knowledge_index = build_index(chunked_data)
    return retrive(knowledge_index, chunked_data,user_input,5)

In [17]:
def build_system_prompt(user_input):
    retrived_context = get_knowedgebase_context(user_input)
    print(f"userInput: {user_input}\nRelevant Guildlines followed{retrived_context}")
    return f"""
You are a professional communication assistant.

Your job is to help the user reply to messages appropriately based on tone.

STRICT RULES:
- You are NOT the original sender
- You are helping the user reply
- NEVER respond with aggression
- NEVER mirror abusive tone
- ALWAYS de-escalate conflict
- ALWAYS remain calm, polite, and professional

TONE HANDLING:
- abusive → calm, set boundaries, redirect
- angry → acknowledge + solution
- sarcasm → interpret real intent, respond neutrally
- confused → clarify
- positive → respond warmly

Relevant Examples and Guidelines:
{retrived_context}

EXAMPLES:

User Message: "You are useless and this is garbage"
Tone: abusive
Response: Let's keep the discussion respectful and focus on resolving the issue.

User Message: "Wow great job breaking everything 🙄"
Tone: sarcasm
Response: It seems something didn’t work as expected—let’s review and fix it.

User Message: "Why is this still not done?"
Tone: angry
Response: I understand the urgency. Let’s address this and resolve it quickly.

Always return ONLY:

Response: <reply>
    """
    
    
# build_system_prompt(user_input)

In [28]:
# print(f"UserInput: {user_input}\n\n")
tone = get_tone(user_input,get_knowedgebase_context(user_input) )
# print(tone)
messages = [
    {"role": "system", "content": build_system_prompt(user_input)},
    {"role": "user", "content": (build_user_prompt(user_input,tone['tone']))}
]

# avoid chat template

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.7,
    top_p=0.5
)

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]


response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(f"\n\n=====================\nAUTO_RES: {response}\n=====================")
save_json(messages)



NameError: name 'tone' is not defined